# Primer estudio de simulación

Este notebook deja solo el flujo principal. Las funciones viven en `src/codispersion_bootstrap/`.

## 0. Instalación local en VS Code

Ejecuta esta celda una vez si aparece `ModuleNotFoundError`. Después reinicia el kernel y continúa con el notebook.

In [1]:
from pathlib import Path
import sys
import subprocess

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
ROOT = None
for path in candidates:
    if (path / 'pyproject.toml').exists() and (path / 'src' / 'codispersion_bootstrap').exists():
        ROOT = path
        break

if ROOT is None:
    raise RuntimeError(f'No encontré la carpeta raíz del proyecto. Estoy parado en: {cwd}')

print('Carpeta raíz encontrada:', ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
print('Paquete instalado correctamente. Reinicia el kernel antes de seguir.')

Carpeta raíz encontrada: C:\ONEDRIVECOPIA\OneDrive - Universidad Técnica Federico Santa María\Escritorio\Memoria\Documentos Necesarios\codispersion_github_template
Paquete instalado correctamente. Reinicia el kernel antes de seguir.


## 1. Imports

In [1]:
import pandas as pd

from codispersion_bootstrap import ExperimentConfig, run_min_experiment
from codispersion_bootstrap.experiments import save_results_json

## 2. Configuración del escenario

In [3]:
cfg = ExperimentConfig(
    n1=256,
    n2=256,
    rho0=0.8,
    range_pix=7,
    mode='aniso',
    angle_deg=45.0,
    hmax=2,
    b=57,
    B=200,   # usar 200--800+ en corrida final
    seed=2025,
)
cfg

ExperimentConfig(n1=256, n2=256, rho0=0.8, range_pix=7, mode='aniso', angle_deg=45.0, hmax=2, b=57, B=200, seed=2025, use_tukey=True, k_px=8, sampler='CBB', mode_rho='toroidal')

## 3. Ejecutar experimento

In [4]:
result = run_min_experiment(cfg)
result.keys()

dict_keys(['params', 'H', 'rho_hat', 'boot'])

## 4. Tabla de codispersión por lag

In [5]:
rho_df = (
    pd.DataFrame.from_dict(result['rho_hat'], orient='index', columns=['rho_hat'])
    .rename_axis('h')
    .reset_index()
)
rho_df

,h,rho_hat
0,"(1, 0)",0.782831
1,"(0, 1)",0.771585
2,"(1, 1)",0.785720
3,"(-1, 1)",0.769468
4,"(2, 0)",0.782479
5,"(0, 2)",0.771496
6,"(2, 2)",0.784919
7,"(-2, 2)",0.770159


## 5. Tabla bootstrap

In [6]:
boot_df = (
    pd.DataFrame.from_dict(result['boot'], orient='index')
    .rename_axis('h')
    .reset_index()
)
boot_df

,h,var_boot,ci_lo,ci_hi
0,"(1, 0)",0.000303,0.731795,0.797490
1,"(0, 1)",0.000321,0.729769,0.795558
2,"(1, 1)",0.000220,0.738170,0.796284
3,"(-1, 1)",0.000169,0.735915,0.783681
4,"(2, 0)",0.000179,0.740655,0.796978
5,"(0, 2)",0.000193,0.740253,0.792437
6,"(2, 2)",0.000171,0.745710,0.797694
7,"(-2, 2)",0.000111,0.743504,0.783827


## 6. Guardar resultados

In [7]:
save_results_json(result, '../results/primer_estudio_resultados.json')